In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema = StructType([
    StructField("country", StringType(), True),
    StructField("total_population", IntegerType(), True)
])

data = [("USA", 1000), ("INDIA", 9999), ("IRAN", 456789), ("INDIA", 10000), ("USA", 1000)]
df = spark.createDataFrame(data, schema=schema)

df.write.mode("overwrite").saveAsTable("test.test.population")

### ---- Built-In SQL Conditional Functions -----

##### CASE functions

In [0]:
%sql
update population set category = 
case
    when total_population > 99999999 then "High"
    when total_population <= 99999999 then "Medium"
end;
alter table population drop column category;
select * from population;

num_affected_rows
4


country,total_population
USA,789456123
CHINA,55555555
AFRICA,99999999
INDIA,331002651


In [0]:
%sql
select  country, 
        total_population,
        case
            when total_population > 99999999 then "High"
            when total_population <= 99999999 then "Medium"
        end as category
from population

country,total_population,category
CHINA,null,null
USA,789456123,High
AFRICA,99999999,Medium
INDIA,331002651,High


In [0]:
from pyspark.sql.functions import when, col
df.display()
df.withColumn(
    "category", 
    when(
        (col("total_population") >= 9999) & (col("total_population") < 100000), 
        "Medium"
    ).when(
        col("total_population") > 10000, 
        "High"
    ).otherwise("Low")).display()


country,total_population
USA,1000
INDIA,9999
IRAN,456789
INDIA,10000
USA,1000


country,total_population,category
USA,1000,Low
INDIA,9999,Medium
IRAN,456789,High
INDIA,10000,Medium
USA,1000,Low


##### NULL function

In [0]:
### NULL --> Used to find out which all records are null in table
### Syntax --> df.filter(col("column_name").isNull()).select("column_name").display()
### Syntax in SQL --> SELECT * FROM table WHERE column_name IS NULL

In [0]:
%sql
-- update population set total_population = null where country = 'CHINA';
select * from population where total_population_2022 is null;


country,total_population_2023,total_population_2022
AFRICA,99999999,null


In [0]:
from pyspark.sql.functions import col, when, lit

df1 = df.withColumn(
    "total_population",
    when(col("country") == "USA", lit(None))
    .otherwise(col("total_population"))
)
df1.display()

df1.filter(col("total_population").isNull()).select("country").display()

country,total_population
USA,null
INDIA,9999
IRAN,456789
INDIA,10000
USA,null


country
USA
USA


In [0]:
%sql
-- alter table population add column total_population_2023 varchar(50);
-- alter table population rename column total_population to total_population_2023;

UPDATE population
SET total_population_2022 =
CASE
    WHEN country = 'CHINA' THEN 1400000000
    WHEN country = 'USA' THEN 780000000
    WHEN country = 'AFRICA' THEN NULL
    WHEN country = 'INDIA' THEN 1300000000
END;

num_affected_rows
4


##### IFNULL function

###### IFNULL --> Used to replace null values of any specific column with default value
###### Syntax --> ifnull(column1, column2) : It means if any record in column1 is null, it will be filled with value of column2

In [0]:
%sql
select  country, 
        total_population_2023, 
        total_population_2022,
        ifnull(total_population_2023, total_population_2022) as total_population
from population;

country,total_population_2023,total_population_2022,total_population
CHINA,null,1400000000,1400000000
USA,789456123,780000000,789456123
AFRICA,99999999,null,99999999
INDIA,331002651,1300000000,331002651


In [0]:
df1.display()

country,total_population
USA,null
INDIA,9999
IRAN,456789
INDIA,10000
USA,null


In [0]:
from pyspark.sql.functions import col, when, coalesce

df2 = df1.withColumnRenamed("total_population", "total_population_2023")
df2.display()

df3 = df2.withColumn("total_population_2022", 
               when(col("country") == "USA", lit(55555555))
               .when(col("country") == "INDIA", lit(37000000))
               .when(col("country") == "IRAN", lit(None))
               .otherwise(col("total_population_2023")))
df3.display()

df3.select(
    col("country"), 
    col("total_population_2023"), 
    col("total_population_2022"), 
    coalesce(
        col("total_population_2023"), 
        col("total_population_2022")
    ).alias("total_population")
).display()

country,total_population_2023
USA,null
INDIA,9999
IRAN,456789
INDIA,10000
USA,null


country,total_population_2023,total_population_2022
USA,null,55555555
INDIA,9999,37000000
IRAN,456789,null
INDIA,10000,37000000
USA,null,55555555


country,total_population_2023,total_population_2022,total_population
USA,null,55555555,55555555
INDIA,9999,37000000,9999
IRAN,456789,null,456789
INDIA,10000,37000000,10000
USA,null,55555555,55555555


##### COALSECE function

In [0]:
%sql
select country,
    total_population_2023,
    total_population_2022,
    coalesce(total_population_2023, total_population_2022) as total_population_coalesce 
from population;

country,total_population_2023,total_population_2022,total_population_coalesce
CHINA,null,1400000000,1400000000
USA,789456123,780000000,789456123
AFRICA,99999999,null,99999999
INDIA,331002651,1300000000,331002651


##### NULLIF function

In [0]:
%sql
select
    country,
    total_population_2023,
    total_population_2022,
    nullif(total_population_2022, 0) as total_population_null
from population;

country,total_population_2023,total_population_2022,total_population_null
CHINA,null,1400000000,1400000000
USA,789456123,780000000,780000000
AFRICA,99999999,null,null
INDIA,331002651,1300000000,1300000000


##### DATE function

In [0]:
%sql
select date('now');
select date_add('2021-03-21', 2);
select date_add(date('now'), -2);
select date_trunc('YEAR', '2021-01-21') as start_of_year;
select current_timestamp;
select year(current_date());
select datediff(current_date(), '2021-01-01') as days_since;

days_since
1957


In [0]:
from pyspark.sql.functions import current_date, date_add, current_timestamp, month, year, date_format

df.display()
df.withColumn("current_date", current_date()) \
    .withColumn("future_date", date_add(current_date(), -2)) \
    .withColumn("time", current_timestamp()) \
    .withColumn("year", year(current_timestamp())) \
    .withColumn("change_format", date_format(date_add(current_timestamp(), -2), "dd/MM/yyyy")) \
.display()

country,total_population
USA,1000
INDIA,9999
IRAN,456789
INDIA,10000
USA,1000


country,total_population,current_date,future_date,time,year,change_format
USA,1000,2026-05-12,2026-05-10,2026-05-12T06:42:27.077Z,2026,10/05/2026
INDIA,9999,2026-05-12,2026-05-10,2026-05-12T06:42:27.077Z,2026,10/05/2026
IRAN,456789,2026-05-12,2026-05-10,2026-05-12T06:42:27.077Z,2026,10/05/2026
INDIA,10000,2026-05-12,2026-05-10,2026-05-12T06:42:27.077Z,2026,10/05/2026
USA,1000,2026-05-12,2026-05-10,2026-05-12T06:42:27.077Z,2026,10/05/2026
